# 03. LOT별로 Z3, Z4, Z5 온도와 출력을 겹쳐 보기

[이슈 #71](https://github.com/posco-3bungi-pdm/pdm-project/issues/71) ,  예측 대상은 `TC-Z3`, `OP-Z3`, `TC-Z4`, `OP-Z4`, `TC-Z5`, `OP-Z5`, `CP-Z4` 총 7개임.

**먼저 온도 3개를 겹쳐 보고, 출력 3개를 겹쳐 봄. 그다음 같은 구역의 온도, 출력을 겹쳐 봄.**

- 가로축: **2024년 실제 측정 날짜, 시각**임. 예를 들어 `01/02 13:26`은 1월 2일 오후 1시 26분임.
- 진한 선: 해당 시각부터 1분 동안의 평균임. 옅은 색: 같은 1분 안의 최소~최대임.
- 온도는 ℃, 출력은 %로 표시함. 두 단위는 데이터사전의 가정임. `CP-Z4`의 단위는 확인되지 않아 기록값으로 표시함.
- 관측이 없는 분은 비워 둠. 원자료와 출력 0을 지우지 않음. 관측 30개 미만을 일괄 삭제하던 과거 임시 기준은 적용하지 않음.
- 현재 자료를 설명하는 그림임. 모델 학습, 시험 분할이나 이상 정답을 만드는 과정은 아님.

`VIEW_LOT`만 바꾸면 화면에 볼 LOT가 달라짐. 전체 14개 LOT의 그림은 매번 같은 파일명으로 저장되므로 중복 파일이 쌓이지 않음. [06번 기준표 생성](06_labeled_dataframe.ipynb)을 먼저 실행할 필요가 없음.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager, rcParams
from IPython.display import display

# 저장소 루트, notebooks 폴더, 개인 노트북 폴더에서 같은 파일을 읽음.
if os.path.isfile(os.path.join('data', 'README.md')):
    ROOT = os.getcwd()
elif os.path.isfile(os.path.join('..', 'data', 'README.md')):
    ROOT = os.path.abspath('..')
else:
    ROOT = os.path.abspath(os.path.join('..', '..'))

DATA_DIR = os.path.join(ROOT, 'data')

TEMP_CSV = os.path.join(DATA_DIR, 'T-CR1-CAL01_온도.csv')
EVENT_CSV = os.path.join(DATA_DIR, 'G-02_조업이벤트.csv')
REPAIR_CSV = os.path.join(DATA_DIR, 'G-01_정기수리캘린더.csv')
EQP_CSV = os.path.join(DATA_DIR, 'G-01_설비마스터.csv')
PART_CSV = os.path.join(DATA_DIR, 'G-01_부품마스터.csv')
MAINT_CSV = os.path.join(DATA_DIR, 'G-02_정비이력.csv')
PARTCHG_CSV = os.path.join(DATA_DIR, 'G-02_부품교체이력.csv')
WELD_CSV = os.path.join(DATA_DIR, 'T-CR1-WLD01_용접.csv')  # 현재 분석에서는 용접 파일을 읽지 않음.
ENC = 'utf-8-sig'

FIG_DIR = os.path.join(ROOT, 'figures', 'namelesspark', '03_lot_comparison')

# CSV를 직접 읽고, 계산에 사용하는 날짜 컬럼을 날짜형으로 바꿈.
temp = pd.read_csv(TEMP_CSV, encoding=ENC)
event = pd.read_csv(EVENT_CSV, encoding=ENC)
repair = pd.read_csv(REPAIR_CSV, encoding=ENC)
eqp = pd.read_csv(EQP_CSV, encoding=ENC)
part = pd.read_csv(PART_CSV, encoding=ENC)
maint = pd.read_csv(MAINT_CSV, encoding=ENC)
partchg = pd.read_csv(PARTCHG_CSV, encoding=ENC)

temp['MEAS_DT'] = pd.to_datetime(temp['MEAS_DT'], errors='raise')
event['EVT_DT'] = pd.to_datetime(event['EVT_DT'], errors='raise')
repair['STRT_DT'] = pd.to_datetime(repair['STRT_DT'], errors='raise')
repair['END_DT'] = pd.to_datetime(repair['END_DT'], errors='raise')
eqp['INSTALL_DT'] = pd.to_datetime(eqp['INSTALL_DT'], errors='raise')
maint['DETC_DT'] = pd.to_datetime(maint['DETC_DT'], errors='raise')
maint['STRT_DT'] = pd.to_datetime(maint['STRT_DT'], errors='raise')
maint['END_DT'] = pd.to_datetime(maint['END_DT'], errors='raise')

# 읽은 원본 데이터프레임을 이름별로 보관함.
sources = {
    'temp': temp,
    'event': event,
    'repair': repair,
    'eqp': eqp,
    'part': part,
    'maint': maint,
    'partchg': partchg,
}

# 설치된 한글 폰트를 사용함.
font_names = []

for installed_font in font_manager.fontManager.ttflist:
    font_names.append(installed_font.name)

for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'NanumBarunGothic', 'Noto Sans CJK KR']:
    if font_name in font_names:
        rcParams['font.family'] = font_name
        break
else:
    print('경고: 사용할 수 있는 한글 폰트를 찾지 못했음. 그래프의 한글이 깨질 수 있음.')

rcParams['axes.unicode_minus'] = False
rcParams['figure.dpi'] = 110
rcParams['axes.spines.top'] = False

TARGETS = ["TC-Z3", "OP-Z3", "TC-Z4", "OP-Z4", "TC-Z5", "OP-Z5", "CP-Z4"]

raw = sources["temp"].sort_values(["LOT_NO", "MEAS_DT"])


In [ ]:
# LOT 관측 범위와 강종을 연결함.
# LOT의 실제 첫 관측과 마지막 관측을 구함.

lot_bounds = raw.groupby("LOT_NO", as_index=False).agg(
    START=("MEAS_DT", "min"),
    END=("MEAS_DT", "max"),
    ROWS=("MEAS_DT", "size"),
)

# CR1 강종변경 중 LOT 첫 관측과 시각이 정확히 같은 기록만 연결함.
lot_grade_changes = event.loc[
    event["PLANT_CD"].eq("CR1") & event["EVT_TYPE"].eq("강종변경"),
    ["EVT_DT", "BEF_VAL", "AFT_VAL"],
].sort_values("EVT_DT")

lot_bounds = lot_bounds.merge(
    lot_grade_changes[["EVT_DT", "AFT_VAL"]],
    left_on="START", right_on="EVT_DT", how="left", validate="many_to_one",
)

lot_bounds = lot_bounds.rename(columns={"AFT_VAL": "GRADE"}).drop(columns="EVT_DT")



In [ ]:
# LOT별 관측 기간과 강종을 정리함.

lot_bounds["DURATION_H"] = (lot_bounds["END"] - lot_bounds["START"]).dt.total_seconds() / 3600

lot_context = lot_bounds[["LOT_NO", "GRADE", "START", "END", "ROWS", "DURATION_H"]]

lot_context = lot_context.sort_values("START").reset_index(drop=True)

lots = lot_context.set_index("LOT_NO")[["START", "END", "GRADE"]]
lots = lots.rename(columns={"START": "시작", "END": "끝", "GRADE": "강종"})


In [ ]:
# LOT별 분 단위 통계를 계산함.

# LOT마다 따로 집계하므로 서로 다른 LOT 사이의 빈 날짜를 메우지 않음
# OP의 0은 실제 출력임. 결측으로 바꾸지 않음
means = {}
lows = {}
highs = {}
counts = {}

for lot, rows in raw.groupby("LOT_NO"):
    minutes = rows.set_index("MEAS_DT")[TARGETS].resample("1min")

    means[lot] = minutes.mean()
    lows[lot] = minutes.min()
    highs[lot] = minutes.max()
    counts[lot] = minutes.count()

# 아래 2개는 팀에서 확정한 해석임. 전체 LOT를 고장으로 표시하는 뜻은 아님
operation = {240058: "정기수리를 위한 정지", 240061: "정기수리 후 재가동"}


## 1. LOT의 강종과 관측 수 확인하기

한 분의 평균이라도 그 안의 실제 관측 수는 다를 수 있음. 마지막 분이 일부만 기록됐을 때도 평균은 남겨 두고 관측 수를 함께 확인함.


In [ ]:
display(lots)

quality = []

for lot, count in counts.items():
    quality.append({
        "LOT": lot,
        "강종": lots.loc[lot, "강종"],
        "전체 분 수": len(count),
        "센서별 관측 수의 최솟값": count.min().min(),
        "센서별 관측 수의 최댓값": count.max().max(),
        "관측이 하나도 없는 센서·분 수": count.eq(0).sum().sum(),
    })

display(pd.DataFrame(quality))

## 2. 온도끼리 → 출력끼리 → 같은 구역의 온도, 출력 보기

먼저 위아래 두 그림에서 Z3, Z4, Z5의 선이 함께 움직이는지 보기. 다음 세 그림에서는 같은 구역의 온도, 출력 변화 시점을 보기.

**온도선과 출력선의 교차 위치는 의미가 없음. 왼쪽과 오른쪽 축이 다르므로 변화한 시점을 비교하기 위함임.** 같은 LOT의 Z3, Z4, Z5 온도축 범위는 같음. 다른 LOT끼리는 다음 노트북에서 비교함.

마지막 CP 그림도 같은 실제 날짜, 시각으로 읽음. 옅은 색의 폭이 넓으면 해당 1분 안에서 값이 크게 움직였다는 뜻임.


In [ ]:
# 화면에는 이 LOT만 표시함. 다른 LOT도 파일로는 모두 저장함
VIEW_LOT = 240001
ZONE_COLORS = {3: "#2B70B2", 4: "#D17A25", 5: "#448A65"}

# 같은 LOT의 세 구역을 같은 온도 범위에서 보기 위함
for lot, mean in means.items():
    low, high = lows[lot], highs[lot]
    time = mean.index

    grade = lots.loc[lot, "강종"]

    note = operation.get(lot, "")
    title = ('LOT ' + str(lot) + ', ' + str(grade))

    if note:
        title += (', ' + str(note))

    tc_min = low[["TC-Z3", "TC-Z4", "TC-Z5"]].min().min() - 1
    tc_max = high[["TC-Z3", "TC-Z4", "TC-Z5"]].max().max() + 1

    # 온도끼리, 출력끼리 겹쳐서 구역 차이를 보기 위함
    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, layout="constrained")

    for ax, prefix, unit in zip(axes, ["TC", "OP"], ["온도 (℃)", "출력 (%)"]):
        for zone, color in ZONE_COLORS.items():
            sensor = (str(prefix) + '-Z' + str(zone))
            ax.plot(time, mean[sensor], color=color, linewidth=1, label=sensor)
            ax.fill_between(time, low[sensor], high[sensor], color=color, alpha=0.10)

        ax.set_ylabel(unit)
        ax.legend(loc="upper left", ncol=3)
        ax.grid(alpha=0.15)

    axes[0].set_ylim(tc_min, tc_max)
    axes[1].set_ylim(0, 105)
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
    axes[1].set_xlabel("2024년 실제 측정 날짜, 시각 ,  선: 1분 평균 ,  옅은 색: 그 분의 최소~최대")
    fig.suptitle(title + " | Z3, Z4, Z5 비교")
    output = os.path.join(FIG_DIR, '구역_간_비교', 'LOT별')

    os.makedirs(output, exist_ok=True)

    fig.savefig(os.path.join(output, (str(lot) + '_' + str(grade) + '_온도_출력_비교.png')), dpi=140)

    if lot == VIEW_LOT:
        plt.show()

    plt.close(fig)

    # 온도와 출력은 단위가 다름. 왼쪽 온도축과 오른쪽 출력축으로 구분함
    for zone in [3, 4, 5]:
        tc, op = ('TC-Z' + str(zone)), ('OP-Z' + str(zone))
        fig, ax = plt.subplots(figsize=(13, 4), layout="constrained")
        right = ax.twinx()
        ax.plot(time, mean[tc], color="#2B70B2", linewidth=1, label=tc)
        ax.fill_between(time, low[tc], high[tc], color="#2B70B2", alpha=0.13)
        right.plot(time, mean[op], color="#D17A25", linewidth=1, label=op)
        right.fill_between(time, low[op], high[op], color="#D17A25", alpha=0.10)
        ax.set_ylim(tc_min, tc_max)
        right.set_ylim(0, 105)
        ax.set_ylabel("온도 (℃)", color="#2B70B2")
        right.set_ylabel("출력 (%)", color="#D17A25")
        ax.legend(loc="upper left")
        right.legend(loc="upper right")
        ax.grid(alpha=0.15)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
        ax.set_xlabel("2024년 실제 측정 날짜, 시각 ,  선: 1분 평균 ,  옅은 색: 그 분의 최소~최대")
        ax.set_title(title + (' | Z' + str(zone) + ' 온도, 출력'))
        output = os.path.join(FIG_DIR, ('Z' + str(zone) + '_온도_출력'), 'LOT별')

        os.makedirs(output, exist_ok=True)

        fig.savefig(os.path.join(output, (str(lot) + '_' + str(grade) + '_Z' + str(zone) + '_온도_출력.png')), dpi=140)

        if lot == VIEW_LOT:
            plt.show()

        plt.close(fig)

    # CP는 온도, 출력과 단위가 달라 별도 그림으로 같은 시각을 확인함
    fig, ax = plt.subplots(figsize=(13, 3.5), layout="constrained")
    ax.plot(time, mean["CP-Z4"], color="#7559A6", linewidth=1, label="CP-Z4")
    ax.fill_between(time, low["CP-Z4"], high["CP-Z4"], color="#7559A6", alpha=0.15)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
    ax.set_ylabel("CP-Z4 기록값 (단위 미확인)")
    ax.set_xlabel("2024년 실제 측정 날짜, 시각 ,  선: 1분 평균 ,  옅은 색: 그 분의 최소~최대")
    ax.set_title(title + " | CP-Z4")
    ax.grid(alpha=0.15)
    ax.legend()
    output = os.path.join(FIG_DIR, 'CP_기록값', 'LOT별')

    os.makedirs(output, exist_ok=True)

    fig.savefig(os.path.join(output, (str(lot) + '_' + str(grade) + '_CP_Z4.png')), dpi=140)

    if lot == VIEW_LOT:
        plt.show()

    plt.close(fig)

## 3. LOT별로 앞서 확인한 내용 이어서 보기

앞선 상세 검토의 관찰을 보존한 표임. 아래 그림만으로 고장 원인이나 모델 성능이 확정된 것은 아님. 제외 구간과 라벨 검토는 [05번 구간 검토](05_interval_review.ipynb)에서 이어감.

| LOT ,  강종 | 7개 채널에서 확인한 변화와 현재 판단 |
| --- | --- |
| 240001 ,  SGCC | `TC-Z3`가 오른 시기에 `OP-Z3`가 낮아지고, 뒤이어 온도가 내려갈 때 출력이 높아졌으며 Z4에서도 온도 상승, 출력 감소가 보였음. 제어 반응으로 해석할 수 있지만 원인은 확정하지 않았고, SGCC는 이 LOT 하나뿐이므로 LOT 전체를 빼거나 이 범위를 정상 기준으로 확정하지 않음. |
| 240024 ,  SPFH590 | `CP-Z4`가 짧게 낮아진 변화는 여러 원자료와 1분 평균에 남고, 가까운 시각 Z5에서는 온도 상승, 출력 감소가 보였음. 특정 고장은 확정하지 않았음. |
| 240037 ,  SPCE | `CP-Z4`가 크게 오른 뒤 약 0.45로 돌아왔고, 가까운 시각 Z4, Z5 온도가 오르면서 출력이 줄었음. 서로 함께 변했다는 사실만으로 CP를 원인으로 정하지 않았음. |
| 240038 ,  SPCEN | `TC-Z4`가 비슷한 범위여도 `OP-Z4`는 114의 후반 하강 전보다 높았지만 CP 중앙값은 비슷했고, 이것만으로 SPCEN의 필수 출력이나 설비 고장을 확정하지 못했음. 이와 별개로 짧은 CP 상승도 보였음. |
| 240044 ,  SPHC | 후반 `TC-Z3`가 올랐다 내려갈 때 `OP-Z3`가 낮아졌다 높아졌고, 출력이 약 99.37%에서 평평하게 이어졌음. 이 값이 실제 설정 상한인지는 확인되지 않았으며 100% 도달 여부나 큰 변동만으로 LOT 전체를 제외하지 않음. |
| 240058 ,  DP590 | **정기수리를 위해 소둔로를 멈춘 LOT이며 후반 온도 하강은 정지 과정임.** `CP-Z4`는 가열 온도보다 먼저 낮아졌으므로 따로 보고, 온도 하강 자체를 고장으로 붙이거나 나중의 오일씰, O링 교체만으로 CP 저하 원인을 확정하지 않음. |
| 240061 ,  SPHC | **정기수리 후 첫 재가동 LOT이며 초기 온도 상승은 재가동 과정임.** 정기수리 종료 후 061 이전에 중간 강종 변경이 없다는 팀 해석을 유지하고, 온도가 오른 뒤에도 높게 남는 `OP-Z4`와 이후 `OP-Z3`의 0 출력, 온도 왕복은 따로 검토함. |
| 240069 ,  SPHC | `CP-Z4`가 여러 차례 짧게 올랐고 가장 큰 상승 때 `TC-Z4`는 오르고 `OP-Z4`는 낮아졌으며, 이후에는 별도의 `OP-Z3` 0 출력과 온도 왕복도 보였음. 나중의 절연 관련 정비, O링 교체를 CP 고장이나 누설의 증거로 확정하지 않았음. |
| 240091 ,  SPHC | 7개 대상 중 `CP-Z4`의 짧은 이탈과 후반 `OP-Z3` 0 출력, `TC-Z3` 왕복을 각각 검토했음. 출력 0이나 높은 평탄값을 그 자체로 고장 또는 실제 설정 상한이라고 확정하지 않음. |
| 240108 ,  SPFH590 | 11:27의 낮은 `OP-Z3` 뒤 11:29에 `TC-Z3` 최고가 나타났고, `OP-Z4` 수준은 같은 SPFH590인 024와 비교했음. 제어 반응이나 지연으로 해석할 여지는 있지만 소재 이동시간, 원인을 확인한 것은 아님. |
| 240114 ,  SPCEN | 하강 전 `TC-Z4`는 038과 비슷하지만 `OP-Z4`는 낮았고, 후반 가열 온도, 출력이 달라져도 `CP-Z4`는 약 0.45를 유지했음. CP까지 낮아진 058과 동일한 사건으로 묶지 않음. |
| 240117 ,  SPCD | 관측 시작부터 온도가 오른 뒤 정기점검, 상태기반 정비 작업이 겹쳤고, 오후에는 별도의 CP 저하가 나타났음. 작업이 원인이라고 확정하지 않았으며 SPCD는 이 LOT 하나뿐이므로 다른 SPCD LOT와 비교해 확인한 것은 아님. |
| 240121 ,  SPCE | 기존 검토에서 `TC-Z3` 최고는 16:42였고 `CP-Z4` 중앙값은 다른 SPCE LOT와 비슷했지만, 주요 추가 검토 대상은 이번 7개 밖의 냉각, 과시효 센서였음. 다른 채널에서 확인한 변화를 이번 7개의 이상 근거로 그대로 옮겨 붙이지 않음. |
| 240133 ,  SPCE | 15:54 정비 감지 전후 `CP-Z4` 중앙값은 비슷했고 실제 정비는 온도 관측 종료 뒤인 16:54에 시작했음. 감지 시각 이후를 모두 이상으로 붙이거나 관측하지 않은 수리 효과를 설명하지 않음. |

038과 114의 비교값은 각각 해당 LOT를 요약한 사후 비교다. 미래 LOT의 중앙값을 과거 LOT 예측 모델의 기준값으로 사용하지 않는다. 117의 SPCD처럼 비교 LOT가 하나뿐인 강종은 다른 LOT에서의 성능을 확인한 것으로 말하지 않는다.


### 제외 구간과 라벨 검토로 이어가기

이 노트북은 LOT별 센서 움직임을 보는 단계임. 현재 적용할 제외 구간, 정상, 이상, 미확정 판단과 근거는 [05_interval_review.ipynb](05_interval_review.ipynb)에서 같이 확인함.

이전 회귀 실험의 보류 시각과 분할표는 현재 기준과 섞이지 않도록 여기서 반복하지 않음. 과거안은 보존본과 Git 이력에 남아 있음. 제외 표시가 없다고 정상으로 확정하는 것은 아님.


<details>
<summary>시계의 시각과 경과시간, 강종 연결 기준 보기</summary>

240001의 첫 관측은 2024-01-02 03:36임. `13:26`은 오후 1시 26분이며 첫 관측 후 **590분**임. 기존 표의 `OP-Z3 4.30% / TC-Z3 866.84℃`는 13:26:00 이상 13:27:00 미만의 평균임. 13:29의 최고 온도와 다른 시각임. 원자료 한 관측값, 1분 평균, 1분 중앙값을 구분함.

이번 그림의 점은 분의 시작에 표시됨. 예를 들어 13:26 점의 평균은 13:27이 된 뒤 계산할 수 있음. 과거 그림이 분 가운데에 점을 놓았다면 점 위치도 30초 다를 수 있음.

LOT, 강종 연결과 관측 공백 기준은 [01 설비, LOT 연결](01_data_alignment.ipynb)에서 확인하면 됨.

6월 수리 중 기록은 6/11 DP590→SPCC, 6/13 SPCC→DP590이고 6/15의 061 시작에 DP590→SPHC임. 058 정지, 061 첫 재가동이라는 팀 판단을 유지함. 수리 중 더미 강판이 실제로 통과했는지는 이 기록만으로 확인할 수 없음. 059, 060의 온도값을 추정해 채우지 않음.

</details>


<details>
<summary>기존 8개 CSV의 시간 연결 결과 보기</summary>

이전 8개 CSV 비교에서 온도 LOT별 첫~마지막 관측 구간의 합은 약 4.12일이었고, 용접 기록이 있는 날짜는 77일이었음. 두 숫자를 실제 조업일수로 해석하지 않음.

온도 관측 기간과 겹친 용접 접합번호는 82개이며 LOT는 240108, 240114, 240117, 240121이었음. 같은 시간에 기록됐다는 뜻이지 같은 소재가 연결되거나 용접이 온도 변화의 원인이라는 뜻은 아님. 소둔로 도착 시각을 확인하려면 소재 추적 정보가 필요함.

정비, 부품 연결과 실제 관측에 겹친 작업은 [02 정비, 부품 분석](02_maintenance_parts.ipynb)에서 확인함. 작업 구간과 실제 정지 구간을 구분해야 함.

</details>
